Set D: Distractor Sensitivity Evaluation

Research Question:
RQ5: How does quantization affect model robustness to distractor passages?

Evaluation Coverage:
- Clean vs distracted context comparison
- Accuracy degradation under distractor pressure
- Distractor citation rates
- Category-specific distractor effects
- Quantization impact on context filtering

Setup

In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
from typing import Dict, List, Tuple
import re
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

print("Imports complete")

In [ ]:
INPUT_DIR = Path('/kaggle/input/generation-sets')
OUTPUT_DIR = Path('/kaggle/working/set_d_evaluation')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
N_BOOTSTRAP = 1000
CONFIDENCE_LEVEL = 0.95
ALPHA = 0.05

np.random.seed(RANDOM_SEED)

print(f"Input: {INPUT_DIR}")
print(f"Output: {OUTPUT_DIR}")

Normalization and Metrics

In [ ]:
def normalize_answer(text: str) -> str:
    """Normalize text following SQuAD evaluation protocol"""
    import unicodedata
    
    if not text:
        return ""
    
    text = unicodedata.normalize('NFD', text)
    text = ''.join(c for c in text if unicodedata.category(c) != 'Mn')
    text = text.lower()
    text = re.sub(r'\b(a|an|the)\b', ' ', text)
    text = re.sub(r'[^a-z0-9\s]', '', text)
    text = ' '.join(text.split())
    
    return text.strip()

def extract_conservative(text: str, ground_truth: str = None, max_ratio: float = 3.0) -> str:
    """Conservative extraction strategy"""
    if not text:
        return ""
    
    text = text.strip()
    
    prefixes = ['Answer:', 'answer:', 'A:', 'a:', 'The answer is:', 'the answer is:']
    for prefix in prefixes:
        if text.lower().startswith(prefix.lower()):
            text = text[len(prefix):].strip()
            break
    
    text = text.split('\n')[0].strip()
    
    if ground_truth and text:
        gt_words = len(ground_truth.split())
        text_words = len(text.split())
        
        if gt_words > 0 and text_words > max_ratio * gt_words:
            sentences = re.split(r'[.!?]+', text)
            if sentences and sentences[0].strip():
                text = sentences[0].strip()
    
    return text.strip()

def exact_match(prediction: str, references: List[str]) -> float:
    """Exact match metric"""
    if not references:
        return 0.0
    
    pred = normalize_answer(prediction)
    refs = [normalize_answer(r) for r in references]
    
    return float(any(pred == ref for ref in refs))

def token_f1(prediction: str, references: List[str]) -> float:
    """Token-level F1 score"""
    if not references:
        return 0.0
    
    pred_tokens = normalize_answer(prediction).split()
    ref_token_lists = [normalize_answer(r).split() for r in references]
    
    if not pred_tokens:
        return 0.0
    
    max_f1 = 0.0
    
    for ref_tokens in ref_token_lists:
        if not ref_tokens:
            continue
        
        common = set(pred_tokens) & set(ref_tokens)
        
        if not common:
            continue
        
        precision = len(common) / len(pred_tokens)
        recall = len(common) / len(ref_tokens)
        
        f1 = 2 * precision * recall / (precision + recall)
        max_f1 = max(max_f1, f1)
    
    return float(max_f1)

def substring_match(prediction: str, references: List[str]) -> float:
    """Substring match metric"""
    if not references:
        return 0.0
    
    pred = normalize_answer(prediction)
    refs = [normalize_answer(r) for r in references]
    
    return float(any(ref in pred for ref in refs if ref))

print("Normalization and metrics defined")

Distractor Detection

In [ ]:
def detect_distractor_citation(
    prediction: str,
    clean_context: str,
    distracted_context: str
) -> Dict:
    """
    Detect if prediction contains information from distractor passages.
    Simple heuristic: check for tokens in prediction that appear in 
    distracted context but not in clean context.
    """
    if not prediction or not clean_context or not distracted_context:
        return {
            'has_distractor_content': False,
            'distractor_token_overlap': 0.0,
            'clean_token_overlap': 0.0
        }
    
    pred_tokens = set(normalize_answer(prediction).split())
    clean_tokens = set(normalize_answer(clean_context).split())
    distracted_tokens = set(normalize_answer(distracted_context).split())
    
    distractor_only_tokens = distracted_tokens - clean_tokens
    
    if not pred_tokens:
        return {
            'has_distractor_content': False,
            'distractor_token_overlap': 0.0,
            'clean_token_overlap': 0.0
        }
    
    pred_distractor_overlap = len(pred_tokens & distractor_only_tokens)
    pred_clean_overlap = len(pred_tokens & clean_tokens)
    
    distractor_ratio = pred_distractor_overlap / len(pred_tokens)
    clean_ratio = pred_clean_overlap / len(pred_tokens)
    
    has_distractor = distractor_ratio > 0.2
    
    return {
        'has_distractor_content': bool(has_distractor),
        'distractor_token_overlap': float(distractor_ratio),
        'clean_token_overlap': float(clean_ratio)
    }

def compute_answer_similarity(pred1: str, pred2: str) -> float:
    """Compute token overlap between two predictions"""
    if not pred1 or not pred2:
        return 0.0
    
    tokens1 = set(normalize_answer(pred1).split())
    tokens2 = set(normalize_answer(pred2).split())
    
    if not tokens1 or not tokens2:
        return 0.0
    
    intersection = len(tokens1 & tokens2)
    union = len(tokens1 | tokens2)
    
    return float(intersection / union) if union > 0 else 0.0

print("Distractor detection functions defined")

Statistical Utilities

In [ ]:
def bootstrap_ci(data: List[float], n_bootstrap: int = 1000, confidence: float = 0.95) -> Tuple[float, float, float]:
    """Bootstrap confidence intervals"""
    data = np.array(data)
    
    if data.size == 0:
        return 0.0, 0.0, 0.0
    
    if len(data) == 1:
        val = float(data[0])
        return val, val, val
    
    bootstrap_means = []
    for _ in range(n_bootstrap):
        sample = np.random.choice(data, size=len(data), replace=True)
        bootstrap_means.append(np.mean(sample))
    
    alpha = 1 - confidence
    lower = np.percentile(bootstrap_means, alpha/2 * 100)
    upper = np.percentile(bootstrap_means, (1 - alpha/2) * 100)
    
    return float(np.mean(data)), float(lower), float(upper)

def cohens_d(group1: np.ndarray, group2: np.ndarray) -> float:
    """Cohen's d effect size"""
    n1, n2 = len(group1), len(group2)
    var1, var2 = np.var(group1, ddof=1), np.var(group2, ddof=1)
    pooled_std = np.sqrt(((n1 - 1) * var1 + (n2 - 1) * var2) / (n1 + n2 - 2))
    return (np.mean(group1) - np.mean(group2)) / pooled_std if pooled_std > 0 else 0.0

def save_json(data: Dict, path: Path):
    """Save JSON with type conversion"""
    def convert(obj):
        if isinstance(obj, (np.integer, np.int64)):
            return int(obj)
        if isinstance(obj, (np.floating, np.float64)):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        if isinstance(obj, (np.bool_, bool)):
            return bool(obj)
        if isinstance(obj, dict):
            return {key: convert(value) for key, value in obj.items()}
        if isinstance(obj, (list, tuple)):
            return [convert(item) for item in obj]
        return obj
    
    with open(path, 'w') as f:
        json.dump(convert(data), f, indent=2)

print("Statistical utilities defined")

Load Generated Data

In [ ]:
def load_set_d_data(config_name: str) -> Dict:
    """Load Set D data for a configuration"""
    file_path = INPUT_DIR / f"{config_name}_set_d_complete.json"
    
    if not file_path.exists():
        print(f"WARNING: {file_path.name} not found")
        return None
    
    with open(file_path) as f:
        return json.load(f)

configs = [
    'fp16_base',
    'fp16_instruct',
    'awq_base',
    'awq_instruct',
    'nf4_base',
    'nf4_instruct',
    'gptq_base',
    'gptq_instruct'
]

data = {}
for config in configs:
    loaded = load_set_d_data(config)
    if loaded:
        data[config] = loaded
        print(f"Loaded {config}: {len(loaded['samples'])} samples")

print(f"\nTotal configs loaded: {len(data)}")

Compute Sample Metrics

In [ ]:
def compute_sample_metrics(sample: Dict) -> Dict:
    """Compute comprehensive metrics for a single sample"""
    ground_truth = sample.get('ground_truth', '')
    ground_truth_variants = sample.get('ground_truth_variants', [])
    if not ground_truth_variants:
        ground_truth_variants = [ground_truth] if ground_truth else []
    
    clean_raw = sample.get('clean_prediction', '')
    distracted_raw = sample.get('distracted_prediction', '')
    
    clean_extracted = extract_conservative(clean_raw, ground_truth)
    distracted_extracted = extract_conservative(distracted_raw, ground_truth)
    
    clean_metrics = {
        'exact_match': exact_match(clean_extracted, ground_truth_variants),
        'token_f1': token_f1(clean_extracted, ground_truth_variants),
        'substring_match': substring_match(clean_extracted, ground_truth_variants)
    }
    
    distracted_metrics = {
        'exact_match': exact_match(distracted_extracted, ground_truth_variants),
        'token_f1': token_f1(distracted_extracted, ground_truth_variants),
        'substring_match': substring_match(distracted_extracted, ground_truth_variants)
    }
    
    accuracy_drop = {
        'exact_match_drop': clean_metrics['exact_match'] - distracted_metrics['exact_match'],
        'token_f1_drop': clean_metrics['token_f1'] - distracted_metrics['token_f1'],
        'substring_match_drop': clean_metrics['substring_match'] - distracted_metrics['substring_match']
    }
    
    distractor_info = detect_distractor_citation(
        distracted_extracted,
        sample.get('clean_context', ''),
        sample.get('distracted_context', '')
    )
    
    answer_consistency = compute_answer_similarity(clean_extracted, distracted_extracted)
    
    return {
        'clean': clean_metrics,
        'distracted': distracted_metrics,
        'accuracy_drop': accuracy_drop,
        'distractor_detection': distractor_info,
        'answer_consistency': answer_consistency,
        'clean_prediction': clean_extracted,
        'distracted_prediction': distracted_extracted,
        'ground_truth': ground_truth
    }

print("Sample metrics function defined")

Aggregate Metrics

In [ ]:
def aggregate_metrics(samples_metrics: List[Dict]) -> Dict:
    """Aggregate metrics across all samples"""
    
    clean_collections = defaultdict(list)
    distracted_collections = defaultdict(list)
    drop_collections = defaultdict(list)
    distractor_collections = defaultdict(list)
    consistency_values = []
    
    for sm in samples_metrics:
        for metric_name in ['exact_match', 'token_f1', 'substring_match']:
            clean_collections[metric_name].append(sm['clean'][metric_name])
            distracted_collections[metric_name].append(sm['distracted'][metric_name])
            drop_collections[f'{metric_name}_drop'].append(sm['accuracy_drop'][f'{metric_name}_drop'])
        
        distractor_collections['has_distractor_content'].append(
            sm['distractor_detection']['has_distractor_content']
        )
        distractor_collections['distractor_token_overlap'].append(
            sm['distractor_detection']['distractor_token_overlap']
        )
        distractor_collections['clean_token_overlap'].append(
            sm['distractor_detection']['clean_token_overlap']
        )
        
        consistency_values.append(sm['answer_consistency'])
    
    results = {
        'clean': {},
        'distracted': {},
        'accuracy_drop': {},
        'distractor_analysis': {},
        'consistency': {}
    }
    
    for metric_name in ['exact_match', 'token_f1', 'substring_match']:
        clean_values = clean_collections[metric_name]
        mean, ci_lower, ci_upper = bootstrap_ci(clean_values, N_BOOTSTRAP, CONFIDENCE_LEVEL)
        
        results['clean'][metric_name] = {
            'mean': float(mean),
            'std': float(np.std(clean_values)),
            'median': float(np.median(clean_values)),
            'ci_lower': float(ci_lower),
            'ci_upper': float(ci_upper)
        }
        
        distracted_values = distracted_collections[metric_name]
        mean, ci_lower, ci_upper = bootstrap_ci(distracted_values, N_BOOTSTRAP, CONFIDENCE_LEVEL)
        
        results['distracted'][metric_name] = {
            'mean': float(mean),
            'std': float(np.std(distracted_values)),
            'median': float(np.median(distracted_values)),
            'ci_lower': float(ci_lower),
            'ci_upper': float(ci_upper)
        }
        
        drop_values = drop_collections[f'{metric_name}_drop']
        mean, ci_lower, ci_upper = bootstrap_ci(drop_values, N_BOOTSTRAP, CONFIDENCE_LEVEL)
        
        results['accuracy_drop'][metric_name] = {
            'mean_drop': float(mean),
            'std_drop': float(np.std(drop_values)),
            'median_drop': float(np.median(drop_values)),
            'ci_lower': float(ci_lower),
            'ci_upper': float(ci_upper)
        }
        
        t_stat, p_value = stats.ttest_rel(clean_values, distracted_values)
        effect_size = cohens_d(np.array(clean_values), np.array(distracted_values))
        
        results['accuracy_drop'][metric_name].update({
            't_statistic': float(t_stat),
            'p_value': float(p_value),
            'significant': bool(p_value < ALPHA),
            'cohens_d': float(effect_size)
        })
    
    results['distractor_analysis'] = {
        'citation_rate': float(np.mean(distractor_collections['has_distractor_content'])),
        'mean_distractor_overlap': float(np.mean(distractor_collections['distractor_token_overlap'])),
        'mean_clean_overlap': float(np.mean(distractor_collections['clean_token_overlap'])),
        'std_distractor_overlap': float(np.std(distractor_collections['distractor_token_overlap'])),
        'std_clean_overlap': float(np.std(distractor_collections['clean_token_overlap']))
    }
    
    mean, ci_lower, ci_upper = bootstrap_ci(consistency_values, N_BOOTSTRAP, CONFIDENCE_LEVEL)
    results['consistency'] = {
        'mean_answer_consistency': float(mean),
        'std_answer_consistency': float(np.std(consistency_values)),
        'ci_lower': float(ci_lower),
        'ci_upper': float(ci_upper)
    }
    
    return results

print("Aggregation function defined")

Compute Complete Metrics

In [ ]:
print("Computing comprehensive metrics for all configurations...")
results = {}

for config_name, config_data in data.items():
    print(f"  {config_name}")
    
    samples = config_data['samples']
    samples_metrics = [compute_sample_metrics(sample) for sample in samples]
    
    results[config_name] = {
        'aggregate': aggregate_metrics(samples_metrics),
        'num_samples': len(samples)
    }

print("Metrics computed")

Quantization Impact Analysis

In [ ]:
def compute_quantization_impact(results: Dict) -> Dict:
    """Compare quantized models vs FP16 for distractor robustness"""
    
    variants = ['base', 'instruct']
    quant_methods = ['awq', 'nf4', 'gptq']
    
    impact_analysis = {}
    
    for variant in variants:
        fp16_config = f'fp16_{variant}'
        
        if fp16_config not in results:
            continue
        
        impact_analysis[variant] = {}
        
        fp16_metrics = results[fp16_config]['aggregate']
        
        for quant_method in quant_methods:
            quant_config = f'{quant_method}_{variant}'
            
            if quant_config not in results:
                continue
            
            quant_metrics = results[quant_config]['aggregate']
            
            impact_analysis[variant][quant_method] = {}
            
            for metric in ['exact_match', 'token_f1']:
                fp16_drop = fp16_metrics['accuracy_drop'][metric]['mean_drop']
                quant_drop = quant_metrics['accuracy_drop'][metric]['mean_drop']
                
                additional_drop = quant_drop - fp16_drop
                
                fp16_clean = fp16_metrics['clean'][metric]['mean']
                quant_clean = quant_metrics['clean'][metric]['mean']
                
                fp16_distracted = fp16_metrics['distracted'][metric]['mean']
                quant_distracted = quant_metrics['distracted'][metric]['mean']
                
                impact_analysis[variant][quant_method][metric] = {
                    'fp16_accuracy_drop': float(fp16_drop),
                    'quant_accuracy_drop': float(quant_drop),
                    'additional_drop_from_quantization': float(additional_drop),
                    'fp16_clean_accuracy': float(fp16_clean),
                    'quant_clean_accuracy': float(quant_clean),
                    'fp16_distracted_accuracy': float(fp16_distracted),
                    'quant_distracted_accuracy': float(quant_distracted)
                }
            
            fp16_citation = fp16_metrics['distractor_analysis']['citation_rate']
            quant_citation = quant_metrics['distractor_analysis']['citation_rate']
            
            impact_analysis[variant][quant_method]['distractor_citation'] = {
                'fp16_citation_rate': float(fp16_citation),
                'quant_citation_rate': float(quant_citation),
                'citation_rate_increase': float(quant_citation - fp16_citation)
            }
    
    return impact_analysis

print("Computing quantization impact analysis...")
quantization_impact = compute_quantization_impact(results)
print("Quantization impact analysis complete")

Robustness Ranking

In [ ]:
def compute_robustness_rankings(results: Dict) -> List[Tuple[str, float]]:
    """Rank models by distractor robustness (lower drop = better)"""
    
    rankings = []
    
    for config_name, config_results in results.items():
        f1_drop = config_results['aggregate']['accuracy_drop']['token_f1']['mean_drop']
        rankings.append((config_name, f1_drop))
    
    rankings.sort(key=lambda x: x[1])
    
    return rankings

print("Computing robustness rankings...")
robustness_rankings = compute_robustness_rankings(results)
print("Rankings computed")

Results Summary: Overall Performance

In [ ]:
print("SET D EVALUATION: DISTRACTOR SENSITIVITY\n")
print("CLEAN VS DISTRACTED CONTEXT PERFORMANCE\n")

print(f"{'Config':<20} {'Clean F1':<12} {'Distract F1':<12} {'F1 Drop':<12} {'Citation %':<12}")

for config_name in configs:
    if config_name not in results:
        continue
    
    agg = results[config_name]['aggregate']
    
    clean_f1 = agg['clean']['token_f1']['mean']
    distracted_f1 = agg['distracted']['token_f1']['mean']
    f1_drop = agg['accuracy_drop']['token_f1']['mean_drop']
    citation_rate = agg['distractor_analysis']['citation_rate'] * 100
    
    print(f"{config_name:<20} {clean_f1:<12.4f} {distracted_f1:<12.4f} {f1_drop:<+12.4f} {citation_rate:<12.2f}")

print("\n\nInterpretation:")
print("  Clean F1: Baseline accuracy without distractors")
print("  Distract F1: Accuracy with distractor passages")
print("  F1 Drop: Performance degradation (positive = worse with distractors)")
print("  Citation %: Percentage of samples citing distractor content")

Results Summary: Statistical Significance

In [ ]:
print("\n\nSTATISTICAL SIGNIFICANCE OF ACCURACY DROP\n")

print(f"{'Config':<20} {'EM Drop':<12} {'F1 Drop':<12} {'t-stat':<12} {'p-value':<12} {'Sig':<5}")

for config_name in configs:
    if config_name not in results:
        continue
    
    agg = results[config_name]['aggregate']
    
    em_drop = agg['accuracy_drop']['exact_match']['mean_drop']
    f1_drop = agg['accuracy_drop']['token_f1']['mean_drop']
    t_stat = agg['accuracy_drop']['token_f1']['t_statistic']
    p_value = agg['accuracy_drop']['token_f1']['p_value']
    
    sig = '***' if p_value < 0.001 else '**' if p_value < 0.01 else '*' if p_value < 0.05 else ''
    
    print(f"{config_name:<20} {em_drop:<+12.4f} {f1_drop:<+12.4f} {t_stat:<+12.4f} {p_value:<12.6f} {sig:<5}")

print("\n\nInterpretation:")
print("  Positive drop values indicate degradation with distractors")
print("  Significant p-values (p < 0.05) confirm distractors impact performance")
print("  Effect sizes (Cohen's d) measure magnitude of degradation")

Results Summary: Distractor Analysis

In [ ]:
print("\n\nDISTRACTOR CONTENT ANALYSIS\n")

print(f"{'Config':<20} {'Citation %':<15} {'Distractor Ovlp':<18} {'Clean Ovlp':<15} {'Consistency':<12}")

for config_name in configs:
    if config_name not in results:
        continue
    
    agg = results[config_name]['aggregate']
    
    citation_rate = agg['distractor_analysis']['citation_rate'] * 100
    distractor_overlap = agg['distractor_analysis']['mean_distractor_overlap']
    clean_overlap = agg['distractor_analysis']['mean_clean_overlap']
    consistency = agg['consistency']['mean_answer_consistency']
    
    print(f"{config_name:<20} {citation_rate:<15.2f} {distractor_overlap:<18.4f} {clean_overlap:<15.4f} {consistency:<12.4f}")

print("\n\nInterpretation:")
print("  Citation %: Models citing distractor content in answers")
print("  Distractor Overlap: Token overlap with distractor-only content")
print("  Clean Overlap: Token overlap with original clean context")
print("  Consistency: Similarity between clean and distracted answers (higher = more stable)")

Results Summary: Quantization Impact

In [ ]:
print("\n\nQUANTIZATION IMPACT ON DISTRACTOR ROBUSTNESS\n")

for variant in ['base', 'instruct']:
    if variant not in quantization_impact:
        continue
    
    print(f"\n{variant.upper()} VARIANT:")
    print(f"{'Quant':<10} {'Metric':<15} {'FP16 Drop':<12} {'Quant Drop':<12} {'Add. Drop':<12}")
    
    for quant_method in ['awq', 'nf4', 'gptq']:
        if quant_method not in quantization_impact[variant]:
            continue
        
        for metric in ['exact_match', 'token_f1']:
            comp = quantization_impact[variant][quant_method][metric]
            
            fp16_drop = comp['fp16_accuracy_drop']
            quant_drop = comp['quant_accuracy_drop']
            additional = comp['additional_drop_from_quantization']
            
            print(f"{quant_method:<10} {metric:<15} {fp16_drop:<+12.4f} {quant_drop:<+12.4f} {additional:<+12.4f}")

print("\n\nCITATION RATE CHANGES:")
print(f"{'Variant':<10} {'Quant':<10} {'FP16 Rate':<12} {'Quant Rate':<12} {'Increase':<12}")

for variant in ['base', 'instruct']:
    if variant not in quantization_impact:
        continue
    
    for quant_method in ['awq', 'nf4', 'gptq']:
        if quant_method not in quantization_impact[variant]:
            continue
        
        cite = quantization_impact[variant][quant_method]['distractor_citation']
        
        fp16_rate = cite['fp16_citation_rate'] * 100
        quant_rate = cite['quant_citation_rate'] * 100
        increase = cite['citation_rate_increase'] * 100
        
        print(f"{variant:<10} {quant_method:<10} {fp16_rate:<12.2f} {quant_rate:<12.2f} {increase:<+12.2f}")

print("\n\nInterpretation:")
print("  Additional Drop: Extra degradation from quantization beyond FP16")
print("  Positive values = quantization hurts distractor robustness")
print("  Citation Rate Increase: Higher rates suggest worse context filtering")

Results Summary: Robustness Rankings

In [ ]:
print("\n\nROBUSTNESS RANKINGS")
print("Models ranked by resistance to distractors (lower F1 drop = better)\n")

print(f"{'Rank':<6} {'Config':<20} {'F1 Drop':<12} {'Rating':<15}")

for i, (config_name, f1_drop) in enumerate(robustness_rankings, 1):
    if abs(f1_drop) < 0.05:
        rating = "Highly Robust"
    elif abs(f1_drop) < 0.10:
        rating = "Robust"
    elif abs(f1_drop) < 0.15:
        rating = "Moderate"
    else:
        rating = "Vulnerable"
    
    print(f"{i:<6} {config_name:<20} {f1_drop:<+12.4f} {rating:<15}")

Detailed Breakdown: Accuracy Metrics

In [ ]:
print("\n\nDETAILED ACCURACY BREAKDOWN\n")

for metric_name in ['exact_match', 'token_f1', 'substring_match']:
    print(f"\n{metric_name.upper().replace('_', ' ')}:")
    print(f"{'Config':<20} {'Clean':<12} {'Distracted':<12} {'Drop':<12} {'Cohen d':<12}")
    
    for config_name in configs:
        if config_name not in results:
            continue
        
        agg = results[config_name]['aggregate']
        
        clean = agg['clean'][metric_name]['mean']
        distracted = agg['distracted'][metric_name]['mean']
        drop = agg['accuracy_drop'][metric_name]['mean_drop']
        cohens = agg['accuracy_drop'][metric_name]['cohens_d']
        
        print(f"{config_name:<20} {clean:<12.4f} {distracted:<12.4f} {drop:<+12.4f} {cohens:<+12.4f}")

Statistical Tests

In [ ]:
print("\n\nSTATISTICAL TESTS")
print("Comparing FP16 vs quantized distractor robustness\n")

for variant in ['base', 'instruct']:
    fp16_config = f'fp16_{variant}'
    
    if fp16_config not in data:
        continue
    
    print(f"\n{variant.upper()} VARIANT (Token F1 Drop):")
    print(f"{'Comparison':<30} {'t-stat':<12} {'p-value':<12} {'Cohen d':<12} {'Sig':<5}")
    
    fp16_samples = []
    for sample in data[fp16_config]['samples']:
        clean_pred = sample.get('clean_prediction', '')
        distracted_pred = sample.get('distracted_prediction', '')
        ground_truth = sample.get('ground_truth', '')
        ground_truth_variants = sample.get('ground_truth_variants', [])
        if not ground_truth_variants:
            ground_truth_variants = [ground_truth] if ground_truth else []
        
        clean_extracted = extract_conservative(clean_pred, ground_truth)
        distracted_extracted = extract_conservative(distracted_pred, ground_truth)
        
        clean_f1 = token_f1(clean_extracted, ground_truth_variants)
        distracted_f1 = token_f1(distracted_extracted, ground_truth_variants)
        
        fp16_samples.append(clean_f1 - distracted_f1)
    
    fp16_samples = np.array(fp16_samples)
    
    for quant_method in ['awq', 'nf4', 'gptq']:
        quant_config = f'{quant_method}_{variant}'
        
        if quant_config not in data:
            continue
        
        quant_samples = []
        for sample in data[quant_config]['samples']:
            clean_pred = sample.get('clean_prediction', '')
            distracted_pred = sample.get('distracted_prediction', '')
            ground_truth = sample.get('ground_truth', '')
            ground_truth_variants = sample.get('ground_truth_variants', [])
            if not ground_truth_variants:
                ground_truth_variants = [ground_truth] if ground_truth else []
            
            clean_extracted = extract_conservative(clean_pred, ground_truth)
            distracted_extracted = extract_conservative(distracted_pred, ground_truth)
            
            clean_f1 = token_f1(clean_extracted, ground_truth_variants)
            distracted_f1 = token_f1(distracted_extracted, ground_truth_variants)
            
            quant_samples.append(clean_f1 - distracted_f1)
        
        quant_samples = np.array(quant_samples)
        
        t_stat, p_value = stats.ttest_ind(fp16_samples, quant_samples)
        effect_size = cohens_d(fp16_samples, quant_samples)
        
        sig = '***' if p_value < 0.001 else '**' if p_value < 0.01 else '*' if p_value < 0.05 else ''
        
        comparison_name = f"FP16 vs {quant_method.upper()}"
        print(f"{comparison_name:<30} {t_stat:<+12.4f} {p_value:<12.6f} {effect_size:<+12.4f} {sig:<5}")

print("\n\nInterpretation:")
print("  Positive effect sizes: Quantized models more vulnerable to distractors")
print("  Significant p-values indicate real differences in robustness")

Save Complete Results

In [ ]:
final_results = {
    'evaluation': 'Set D - Distractor Sensitivity',
    'configs': {
        config_name: {
            'aggregate': config_results['aggregate'],
            'num_samples': config_results['num_samples']
        }
        for config_name, config_results in results.items()
    },
    'quantization_impact': quantization_impact,
    'robustness_rankings': [
        {'rank': i, 'config': name, 'f1_drop': drop}
        for i, (name, drop) in enumerate(robustness_rankings, 1)
    ],
    'metadata': {
        'n_bootstrap': N_BOOTSTRAP,
        'confidence_level': CONFIDENCE_LEVEL,
        'alpha': ALPHA,
        'random_seed': RANDOM_SEED,
        'metrics': {
            'accuracy': ['exact_match', 'token_f1', 'substring_match'],
            'distractor': ['citation_rate', 'distractor_token_overlap', 'clean_token_overlap'],
            'consistency': ['answer_consistency']
        },
        'analyses': ['accuracy_drop', 'distractor_citation', 'quantization_impact', 'robustness_rankings']
    }
}

output_path = OUTPUT_DIR / 'set_d_complete_results.json'
save_json(final_results, output_path)
print(f"\nComplete results saved to: {output_path}")

accuracy_summary = []

for config_name in configs:
    if config_name not in results:
        continue
    
    agg = results[config_name]['aggregate']
    
    row = {
        'config': config_name,
        'clean_em': agg['clean']['exact_match']['mean'],
        'clean_f1': agg['clean']['token_f1']['mean'],
        'distracted_em': agg['distracted']['exact_match']['mean'],
        'distracted_f1': agg['distracted']['token_f1']['mean'],
        'em_drop': agg['accuracy_drop']['exact_match']['mean_drop'],
        'f1_drop': agg['accuracy_drop']['token_f1']['mean_drop'],
        'citation_rate': agg['distractor_analysis']['citation_rate'],
        'answer_consistency': agg['consistency']['mean_answer_consistency']
    }
    
    accuracy_summary.append(row)

accuracy_df = pd.DataFrame(accuracy_summary)
csv_path = OUTPUT_DIR / 'accuracy_summary.csv'
accuracy_df.to_csv(csv_path, index=False)
print(f"Accuracy summary CSV: {csv_path}")

distractor_analysis = []

for config_name in configs:
    if config_name not in results:
        continue
    
    agg = results[config_name]['aggregate']
    
    row = {
        'config': config_name,
        'citation_rate': agg['distractor_analysis']['citation_rate'],
        'mean_distractor_overlap': agg['distractor_analysis']['mean_distractor_overlap'],
        'mean_clean_overlap': agg['distractor_analysis']['mean_clean_overlap'],
        'std_distractor_overlap': agg['distractor_analysis']['std_distractor_overlap'],
        'std_clean_overlap': agg['distractor_analysis']['std_clean_overlap']
    }
    
    distractor_analysis.append(row)

distractor_df = pd.DataFrame(distractor_analysis)
csv_path = OUTPUT_DIR / 'distractor_analysis.csv'
distractor_df.to_csv(csv_path, index=False)
print(f"Distractor analysis CSV: {csv_path}")

quantization_summary = []

for variant in ['base', 'instruct']:
    if variant not in quantization_impact:
        continue
    
    for quant_method in ['awq', 'nf4', 'gptq']:
        if quant_method not in quantization_impact[variant]:
            continue
        
        row = {
            'variant': variant,
            'quant_method': quant_method
        }
        
        for metric in ['exact_match', 'token_f1']:
            comp = quantization_impact[variant][quant_method][metric]
            row[f'{metric}_fp16_drop'] = comp['fp16_accuracy_drop']
            row[f'{metric}_quant_drop'] = comp['quant_accuracy_drop']
            row[f'{metric}_additional_drop'] = comp['additional_drop_from_quantization']
        
        cite = quantization_impact[variant][quant_method]['distractor_citation']
        row['fp16_citation_rate'] = cite['fp16_citation_rate']
        row['quant_citation_rate'] = cite['quant_citation_rate']
        row['citation_rate_increase'] = cite['citation_rate_increase']
        
        quantization_summary.append(row)

quantization_df = pd.DataFrame(quantization_summary)
csv_path = OUTPUT_DIR / 'quantization_impact_summary.csv'
quantization_df.to_csv(csv_path, index=False)
print(f"Quantization impact summary CSV: {csv_path}")

robustness_df = pd.DataFrame(robustness_rankings, columns=['config', 'f1_drop'])
robustness_df['rank'] = range(1, len(robustness_rankings) + 1)
csv_path = OUTPUT_DIR / 'robustness_rankings.csv'
robustness_df.to_csv(csv_path, index=False)
print(f"Robustness rankings CSV: {csv_path}")

print("\nEVALUATION COMPLETE")
print("Set D includes: Distractor Sensitivity, Accuracy Degradation, Citation Analysis, and Quantization Impact")